# Calculus & Linear Algebra Warm-up for Causal Inference
**3-day sprint · derivatives → integrals → linear algebra → probability**

Run every cell top to bottom. By the end you are ready for Blitzstein Week 1.

| Day | Topic | Why it matters |
|-----|-------|----------------|
| 1 | Derivatives | Finding maxima of likelihoods, backprop |
| 2 | Integrals | Computing probabilities, E[X] |
| 3 | Linear Algebra | Regression, OLS, causal estimators, ML models |

---

In [ ]:
# Setup — run this first
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.facecolor'] = '#f8f8f8'

print('Libraries loaded. Ready to go.')

---
## DAY 1 — Derivatives

A derivative measures how fast a function changes at a point. Geometrically: **the slope of the curve**.

$$f'(x) = \\lim_{h \\to 0} \\frac{f(x+h) - f(x)}{h}$$

> **Why this matters for causal inference:** estimating treatment effects means maximising a likelihood. Maximising = finding where the derivative = 0.

**Power rule:** if $f(x) = x^n$ then $f'(x) = n \\cdot x^{n-1}$

In [ ]:
# Cell 1: Power rule verification
# f(x) = 3x^2 + 2x  ->  f'(x) = 6x + 2

def f(x):
    return 3*x**2 + 2*x

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x)) / h

x = 4
df_analytical = 6*x + 2
df_numerical  = numerical_derivative(f, x)

print(f'f(x) = 3x^2 + 2x   at x = {x}')
print(f"f'(x) analytical = {df_analytical}")
print(f"f'(x) numerical  = {df_numerical:.4f}")
print(f'Match? {abs(df_numerical - df_analytical) < 0.01}')

### The chain rule

$$\\frac{d}{dx}[f(g(x))] = f'(g(x)) \\cdot g'(x)$$

> **Why this matters:** backpropagation in neural networks is the chain rule applied recursively.
Also used when differentiating log-likelihoods for causal estimators.

In [ ]:
# Cell 2: Chain rule
# d/dx [log(x^2 + 1)] = 2x / (x^2 + 1)

import math

def g(x):
    return math.log(x**2 + 1)

def g_prime_analytical(x):
    return 2*x / (x**2 + 1)

def g_prime_numerical(x, h=1e-7):
    return (g(x + h) - g(x)) / h

x = 3.0
print('d/dx[log(x^2 + 1)]  at x = 3')
print(f'Analytical (chain rule): {g_prime_analytical(x):.6f}')
print(f'Numerical approx:        {g_prime_numerical(x):.6f}')
print()
print('Log derivatives appear everywhere in log-likelihood estimation.')

In [ ]:
# Cell 3: Finding the maximum where f'(x) = 0
# f(x) = -x^2 + 4x + 1  ->  f'(x) = -2x + 4  ->  max at x = 2

import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return -x**2 + 4*x + 1

def f_prime(x, h=1e-7):
    return (f(x + h) - f(x)) / h

xs = np.linspace(0, 5, 1000)
max_x = xs[np.argmax(f(xs))]

print('f(x) = -x^2 + 4x + 1')
print(f'Analytical max at x = 2       f(2) = {f(2)}')
print(f'Numerical  max at x = {max_x:.3f}    f   = {f(max_x):.4f}')
print(f"f'(2) = {f_prime(2):.8f}   (should be ~0)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
xp = np.linspace(-1, 5, 300)
ax1.plot(xp, f(xp), color='#7c6af7', linewidth=2.5)
ax1.axvline(x=2, color='#f7b267', linestyle='--', label='max at x=2')
ax1.scatter([2], [f(2)], color='#f7b267', zorder=5, s=80)
ax1.set_title('f(x)'); ax1.legend()
ax2.plot(xp, f_prime(xp), color='#4ecdc4', linewidth=2.5)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(x=2, color='#f7b267', linestyle='--', label="f'(x)=0 at x=2")
ax2.set_title("f'(x)"); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Cell 4: Visualise sin(x) and its derivative cos(x)

import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-2*np.pi, 2*np.pi, 500)
plt.figure(figsize=(11, 4))
plt.plot(x, np.sin(x), color='#7c6af7', linewidth=2.5, label='sin(x)')
plt.plot(x, np.cos(x), color='#4ecdc4', linewidth=2, linestyle='--', label="cos(x) = sin'(x)")
plt.axhline(0, color='gray', linewidth=0.8, alpha=0.5)
for px in [-3*np.pi/2, np.pi/2]:
    plt.axvline(px, color='#f7b267', alpha=0.4, linewidth=1)
plt.title('Where sin peaks, cos = 0  (derivative = 0 at maximum)')
plt.xlabel('x'); plt.legend()
plt.tight_layout(); plt.show()

---
## DAY 2 — Integrals

$$\\int_a^b f(x)\\, dx = F(b) - F(a) \\quad \\text{where } F'(x) = f(x)$$

> **Key connection:** $P(a \\leq X \\leq b) = \\int_a^b f(x)\\, dx$
> You will write this every week in Blitzstein.

In [ ]:
# Cell 5: Riemann sums (numerical integration)

import numpy as np

def riemann_integral(f, a, b, n=100_000):
    h = (b - a) / n
    xs = np.linspace(a, b - h, n)
    return np.sum(f(xs) * h)

r1 = riemann_integral(lambda x: 2*x, 0, 1)
r2 = riemann_integral(lambda x: x**2, 0, 2)

print('integral from 0 to 1 of 2x dx')
print(f'  Analytical: 1.000000')
print(f'  Numerical:  {r1:.6f}')
print()
print('integral from 0 to 2 of x^2 dx')
print(f'  Analytical: {8/3:.6f}')
print(f'  Numerical:  {r2:.6f}')

In [ ]:
# Cell 6: Integrals = Probabilities

import numpy as np
import matplotlib.pyplot as plt

def riemann_integral(f, a, b, n=100_000):
    h = (b - a) / n
    xs = np.linspace(a, b - h, n)
    return np.sum(f(xs) * h)

def uniform_pdf(x):
    return np.where((x >= 0) & (x <= 1), 1.0, 0.0)

total = riemann_integral(uniform_pdf, 0, 1)
prob  = riemann_integral(uniform_pdf, 0.2, 0.7)
print(f'Total probability: {total:.6f}  (should be 1.0)')
print(f'P(0.2 <= X <= 0.7): {prob:.6f}  (should be 0.5)')

x = np.linspace(-0.5, 1.5, 400)
plt.figure(figsize=(8, 3.5))
plt.plot(x, uniform_pdf(x), color='#7c6af7', linewidth=2.5)
x_s = np.linspace(0.2, 0.7, 200)
plt.fill_between(x_s, uniform_pdf(x_s), alpha=0.35, color='#7c6af7',
                 label='P(0.2<=X<=0.7) = 0.5')
plt.ylim(-0.1, 1.5)
plt.title('Uniform[0,1] -- shaded area = probability')
plt.xlabel('x'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7: Normal distribution and p-values

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def riemann_integral(f, a, b, n=100_000):
    h = (b - a) / n
    xs = np.linspace(a, b - h, n)
    return np.sum(f(xs) * h)

def normal_pdf(x, mu=0, sigma=1):
    return (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x-mu)/sigma)**2)

p1 = riemann_integral(lambda x: normal_pdf(x), -1, 1)
p2 = riemann_integral(lambda x: normal_pdf(x), -2, 2)
p3 = riemann_integral(lambda x: normal_pdf(x), -3, 3)
print(f'P(-1s<=X<=1s) = {p1:.4f}  (expected ~0.6827)')
print(f'P(-2s<=X<=2s) = {p2:.4f}  (expected ~0.9545)')
print(f'P(-3s<=X<=3s) = {p3:.4f}  (expected ~0.9973)')
print('P(|Z|>1.96) ~ 0.05  ->  5% significance level in A/B tests.')

x = np.linspace(-4, 4, 500)
plt.figure(figsize=(10, 4))
plt.plot(x, normal_pdf(x), color='#7c6af7', linewidth=2.5)
for a, b, col, alpha in [(-1,1,'#7c6af7',0.35),(-2,-1,'#4ecdc4',0.25),
                          (1,2,'#4ecdc4',0.25),(-3,-2,'#f7b267',0.2),(2,3,'#f7b267',0.2)]:
    xf = np.linspace(a, b, 200)
    plt.fill_between(xf, normal_pdf(xf), alpha=alpha, color=col)
plt.axvline(-1.96, color='#f7706e', linestyle='--', alpha=0.7)
plt.axvline( 1.96, color='#f7706e', linestyle='--', alpha=0.7)
plt.text(2.0, 0.3, '1.96\n(p=0.05)', color='#f7706e', fontsize=9)
patches = [mpatches.Patch(color='#7c6af7', alpha=0.6, label='+-1s  68.3%'),
           mpatches.Patch(color='#4ecdc4', alpha=0.5, label='+-2s  95.4%'),
           mpatches.Patch(color='#f7b267', alpha=0.5, label='+-3s  99.7%')]
plt.legend(handles=patches, loc='upper left')
plt.title('Normal(0,1) -- the distribution behind p-values')
plt.xlabel('Standard deviations'); plt.tight_layout(); plt.show()

In [ ]:
# Cell 8: E[X] and Var[X] via integration

import numpy as np

def riemann_integral(f, a, b, n=100_000):
    h = (b - a) / n
    xs = np.linspace(a, b - h, n)
    return np.sum(f(xs) * h)

def uniform_pdf(x):
    return np.where((x >= 0) & (x <= 1), 1.0, 0.0)

EX   = riemann_integral(lambda x: x * uniform_pdf(x),    0, 1)
EX2  = riemann_integral(lambda x: x**2 * uniform_pdf(x), 0, 1)
VarX = EX2 - EX**2
print(f'E[X]   = {EX:.6f}   (should be 0.5)')
print(f'Var[X] = {VarX:.6f}  (should be 1/12 = {1/12:.6f})')
print('Day 2 complete.')

---
## DAY 3 — Linear Algebra

Linear algebra is the language of machine learning, regression, and causal estimators.
Almost every model you build writes data as a **matrix** and parameters as a **vector**.

| Concept | Formula | Used for |
|---------|---------|----------|
| Matrix-vector product | $A\\mathbf{x} = \\mathbf{b}$ | Applying a linear model to data |
| OLS estimator | $\\hat{\\beta} = (X^TX)^{-1}X^Ty$ | Regression, ATE estimation |
| Eigendecomposition | $A\\mathbf{v} = \\lambda\\mathbf{v}$ | PCA, understanding covariance |
| SVD | $A = U\\Sigma V^T$ | Dimensionality reduction, embeddings |

> **Rabbit hole warning:** do NOT go into proofs of eigenvalue existence or spectral theorem.
> Learn to *use* these operations. Intuition + code is enough for now.

In [ ]:
# Cell 9: Vectors and dot products
# The dot product is the foundation of every linear model.
# y_hat = w . x  means: weighted sum of features

import numpy as np

# Two vectors
w = np.array([0.5, 1.2, -0.3])   # model weights (coefficients)
x = np.array([1.0, 2.0,  3.0])   # one observation (features)

dot_manual = sum(wi * xi for wi, xi in zip(w, x))
dot_numpy  = np.dot(w, x)

print('w (weights):', w)
print('x (features):', x)
print()
print('Dot product manual:', dot_manual)
print('Dot product numpy: ', dot_numpy)
print()
print('This is y_hat = w . x  -- the prediction of a linear model.')
print('In causal inference: treatment effect = beta . covariates.')

In [ ]:
# Cell 10: Matrix multiplication
# X is your data matrix (n observations x p features)
# beta is your coefficient vector
# y_hat = X @ beta  gives predictions for all observations at once

import numpy as np

# Data matrix: 5 observations, 3 features
np.random.seed(42)
X = np.random.randn(5, 3)
beta = np.array([1.0, -0.5, 0.8])   # true coefficients

y_hat = X @ beta     # matrix-vector product -> predictions

print('Data matrix X (5 obs x 3 features):')
print(np.round(X, 3))
print()
print('Coefficients beta:', beta)
print()
print('Predictions y_hat = X @ beta:')
print(np.round(y_hat, 4))
print()
print('This is the core operation of linear regression and OLS.')

In [ ]:
# Cell 11: OLS estimator -- the heart of regression
# beta_hat = (X'X)^{-1} X'y
# This is the closed-form solution that minimises sum of squared residuals.
# Under randomisation it is also an unbiased estimator of the ATE.

import numpy as np

np.random.seed(0)
n = 200

# Simulate data: y = 2 + 3*x + noise
x      = np.random.randn(n)
noise  = np.random.randn(n) * 0.5
y      = 2 + 3*x + noise

# Design matrix with intercept column
X = np.column_stack([np.ones(n), x])

# OLS formula: beta = (X'X)^{-1} X'y
beta_hat = np.linalg.inv(X.T @ X) @ X.T @ y

print('True coefficients: intercept=2.0, slope=3.0')
print(f'OLS estimates:     intercept={beta_hat[0]:.4f}, slope={beta_hat[1]:.4f}')
print()
print('Formula used: beta_hat = (X^T X)^{-1} X^T y')
print()
print('In causal inference with an RCT:')
print('  X = [1, D]  where D is treatment indicator (0/1)')
print('  beta_hat[1] = estimated Average Treatment Effect (ATE)')

In [ ]:
# Cell 12: Visualise OLS fit

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
n = 200
x     = np.random.randn(n)
noise = np.random.randn(n) * 0.5
y     = 2 + 3*x + noise
X     = np.column_stack([np.ones(n), x])
beta  = np.linalg.inv(X.T @ X) @ X.T @ y

x_line = np.linspace(x.min(), x.max(), 100)
y_line = beta[0] + beta[1] * x_line

plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.4, color='#7c6af7', s=20, label='data')
plt.plot(x_line, y_line, color='#f7b267', linewidth=2.5,
         label=f'OLS fit: y = {beta[0]:.2f} + {beta[1]:.2f}x')
plt.title('OLS regression -- beta_hat = (X^T X)^{-1} X^T y', fontsize=12)
plt.xlabel('x (e.g. treatment dose)')
plt.ylabel('y (e.g. outcome)')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Cell 13: Eigenvalues and eigenvectors (intuition)
# A eigenvector v is a direction that a matrix only stretches, never rotates.
# The eigenvalue lambda is the stretch factor.
# A @ v = lambda * v
#
# Why this matters: PCA (used in causal ML for confounder control)
# is entirely about finding the eigenvectors of the covariance matrix.

import numpy as np

# Covariance matrix of two correlated variables
A = np.array([[3.0, 1.5],
              [1.5, 2.0]])

eigenvalues, eigenvectors = np.linalg.eig(A)

print('Covariance matrix A:')
print(A)
print()
print('Eigenvalues:', np.round(eigenvalues, 4))
print('Eigenvectors (columns):')
print(np.round(eigenvectors, 4))
print()
print('Verify: A @ v = lambda * v')
v0 = eigenvectors[:, 0]
lhs = A @ v0
rhs = eigenvalues[0] * v0
print(f'  A @ v0        = {np.round(lhs, 6)}')
print(f'  lambda0 * v0  = {np.round(rhs, 6)}')
print(f'  Equal?          {np.allclose(lhs, rhs)}')
print()
print('PCA keeps only the eigenvectors with the largest eigenvalues.')
print('This is how you reduce 100 confounders to 5 principal components.')

In [ ]:
# Cell 14: SVD -- Singular Value Decomposition
# A = U * Sigma * V^T
# The most important matrix decomposition in ML.
# Used in: PCA, recommender systems, NLP embeddings, image compression.

import numpy as np

np.random.seed(1)
A = np.random.randn(4, 3)   # 4 observations, 3 features

U, s, Vt = np.linalg.svd(A, full_matrices=False)

print('Original matrix A (4x3):')
print(np.round(A, 3))
print()
print('Singular values:', np.round(s, 4))
print('(Larger = more important direction in the data)')
print()

# Reconstruct A from SVD
A_reconstructed = U @ np.diag(s) @ Vt
print('Reconstruction error (should be ~0):', np.round(np.max(np.abs(A - A_reconstructed)), 10))
print()
print('Low-rank approximation (keep only top 1 singular value):')
A_rank1 = s[0] * np.outer(U[:, 0], Vt[0, :])
print(np.round(A_rank1, 3))
print()
print('This rank-1 approximation captures the main structure of the data.')
print('PCA is SVD applied to the centred data matrix.')

In [ ]:
# Cell 15: OLS via SVD (connecting everything)
# In practice numpy uses SVD to solve OLS -- more stable than inverting X'X.
# This cell shows they give the same answer.

import numpy as np

np.random.seed(0)
n = 200
x     = np.random.randn(n)
y     = 2 + 3*x + np.random.randn(n)*0.5
X     = np.column_stack([np.ones(n), x])

# Method 1: closed-form OLS
beta_ols = np.linalg.inv(X.T @ X) @ X.T @ y

# Method 2: numpy least squares (uses SVD internally)
beta_lstsq, _, _, _ = np.linalg.lstsq(X, y, rcond=None)

print('OLS via (X^T X)^{-1} X^T y :', np.round(beta_ols, 6))
print('OLS via np.linalg.lstsq    :', np.round(beta_lstsq, 6))
print('Identical?', np.allclose(beta_ols, beta_lstsq))
print()
print('='*50)
print('  WARM-UP COMPLETE. Ready for Week 1.  ')
print('='*50)
print()
print('You can now:')
print('  - Compute derivatives  (slopes, maxima, backprop)')
print('  - Compute integrals    (probabilities, E[X])')
print('  - Work with matrices   (OLS, PCA, SVD, regression)')
print()
print('Open Blitzstein ch.1 on Monday.')

---
## Quick reference

### Calculus
| Concept | Formula | Used for |
|---------|---------|----------|
| Derivative | $f'(x) = \\lim_{h\\to0}\\frac{f(x+h)-f(x)}{h}$ | Maxima of likelihoods |
| Power rule | $\\frac{d}{dx}x^n = nx^{n-1}$ | Polynomials |
| Chain rule | $\\frac{d}{dx}f(g(x)) = f'(g(x))\\cdot g'(x)$ | Log-likelihoods, backprop |
| Integral | $\\int_a^b f(x)\\,dx = F(b)-F(a)$ | Probabilities |
| Expectation | $E[X] = \\int x\\,f(x)\\,dx$ | Average treatment effect |

### Linear Algebra
| Concept | Formula | Used for |
|---------|---------|----------|
| Dot product | $\\mathbf{w}\\cdot\\mathbf{x} = \\sum_i w_i x_i$ | Linear model prediction |
| Matrix product | $Y = XB$ | Batch predictions, transformations |
| OLS | $\\hat{\\beta} = (X^TX)^{-1}X^Ty$ | Regression, ATE estimation |
| Eigendecomposition | $A\\mathbf{v} = \\lambda\\mathbf{v}$ | PCA, covariance structure |
| SVD | $A = U\\Sigma V^T$ | PCA, embeddings, stable OLS |

**Next:** Blitzstein & Hwang — *Introduction to Probability*, chapters 1-7.